# G9 雙塔架構（DualTower）Fine-tuning

本 notebook 實作 G9 Siamese 雙塔分類器：title 與 content 各自通過獨立 tokenize（共享 XLM-R backbone），

與 G8 唯一差異在架構：content 有獨立的 256 token budget，不與 title 共搶預算。

- **C1**：不對稱合併向量 `[h_t; h_c; h_t−h_c; h_c−h_t]`（有號方向，區分兌現落差方向）
- **C2**：title-aware attention pooling（以 title pooled 向量為 query，對 content token 加權）
- **C3**（flag）：塔頂 title <-> content 一層 cross-attention 對齊層
- **C4**（flag）：consistency loss 正則項（利用 423 組同內文×不同標題對）

透過 `USE_CROSS_ATTENTION` / `USE_CONSISTENCY_LOSS` flag 切換 Run 1 / Run 2 配置。

## 環境安裝

In [ ]:
# 在 Colab 上安裝所需套件；本機執行時可略過。
!pip install -q transformers datasets accelerate scikit-learn

## 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 路徑與超參數設定

In [ ]:
from pathlib import Path

DRIVE_PROJECT    = Path("/content/drive/MyDrive/NLP_Project")
PROCESSED_DIR    = DRIVE_PROJECT / "dataset" / "processed"

MODEL_NAME    = "xlm-roberta-base"
BATCH_SIZE    = 16
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-5
SEED          = 42

# G9 雙塔專屬設定
TITLE_MAX_LEN      = 64
CONTENT_MAX_LEN    = 256
RUN_TAG            = "g9"
USE_ADVERSARIAL_G8 = True   # G9 沿用 G8 訓練資料（隔離架構變因）
THRESHOLD = 0.50

TOKEN_DROPOUT_PROB = 0.1
CONTENT_CROP_PROB  = 0.5

# ── G9 架構 flag（C3 / C4）────────────────────────────────────────
# C1（不對稱合併）與 C2（title-aware pooling）為 baseline，預設常開。
# C3、C4 用 flag 切換，讓一份程式碼跑出 Run 1 / Run 2 不同配置。
#
# Run 1：USE_CROSS_ATTENTION=True,  USE_CONSISTENCY_LOSS=False
# Run 2（情況 a）：兩者皆 True
# Run 2（情況 b）：USE_CROSS_ATTENTION=False（回退守主 test），C4 維持關
USE_CROSS_ATTENTION    = True   # C3：塔頂 title↔content 一層 cross-attention 對齊層
USE_CONSISTENCY_LOSS   = False  # C4：Run 2 條件性底牌，預設關
CONSISTENCY_LOSS_ALPHA = 0.1    # C4 loss 權重（小，作正則項，避免壓過主任務）

MODEL_OUTPUT_DIR = DRIVE_PROJECT / f"models/xlm-roberta-clickbait-{RUN_TAG}-dualtower"
METRICS_OUT      = DRIVE_PROJECT / f"results/{RUN_TAG}_transformer_metrics.json"

print(f"Run tag: {RUN_TAG}")
print(f"Model output: {MODEL_OUTPUT_DIR}")
print(f"USE_CROSS_ATTENTION={USE_CROSS_ATTENTION}  USE_CONSISTENCY_LOSS={USE_CONSISTENCY_LOSS}")
print(f"Title max len: {TITLE_MAX_LEN} | Content max len: {CONTENT_MAX_LEN}")

## 匯入套件

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 載入資料

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
valid_df = pd.read_csv(PROCESSED_DIR / "valid.csv")
test_df  = pd.read_csv(PROCESSED_DIR / "test.csv")

# G9 沿用 G8 對抗資料，與 G8 訓練資料相同（隔離架構變因）
if USE_ADVERSARIAL_G8:
    adv_path = PROCESSED_DIR / "adversarial_g8.csv"
    adv = pd.read_csv(adv_path)
    train_df = pd.concat([train_df, adv], ignore_index=True)
    print(f"G9 (USE_ADVERSARIAL_G8=True): 併入對抗資料 {len(adv)} 筆，總訓練筆數 {len(train_df)}")

for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    assert set(df.columns) >= {"id", "lang", "title", "content", "label"}, f"{name} missing columns"
    assert set(df["label"].unique()).issubset({0, 1}), f"{name} has unexpected labels"
    assert {"zh", "en"}.issubset(set(df["lang"].unique())), f"{name} missing a language"
    print(f"{name}: {len(df):,} rows | label 0={df['label'].eq(0).sum():,} label 1={df['label'].eq(1).sum():,}")

In [ ]:
# ── 【可選】刪減 tw_neg_only，測 Bug3 已馴服後能否再降 FN ──
# 放在「載入資料」cell 之後執行。改 KEEP_RATIO 一個數即可切 A/B/C/D。
# 1.0=全留(A) / 0.5=B / 0.25=C / 0.0=全刪(D)。不動任何 csv。
TW_NEG_KEEP_RATIO = 0.5

if USE_ADVERSARIAL_G8 and TW_NEG_KEEP_RATIO < 1.0:
    is_twneg = train_df["id"].astype(str).str.endswith("_twneg")
    twneg = train_df[is_twneg]
    keep_n = int(round(len(twneg) * TW_NEG_KEEP_RATIO))
    twneg_keep = twneg.sample(n=keep_n, random_state=SEED) if keep_n > 0 else twneg.iloc[0:0]
    train_df = pd.concat([train_df[~is_twneg], twneg_keep], ignore_index=True)
    print(f"tw_neg_only: {int(is_twneg.sum())} → 保留 {keep_n} (ratio={TW_NEG_KEEP_RATIO})")
    print(f"訓練筆數: {len(train_df)} | label 0={train_df['label'].eq(0).sum()} label 1={train_df['label'].eq(1).sum()}")
else:
    print(f"tw_neg_only 未刪減 (G8={USE_ADVERSARIAL_G8}, ratio={TW_NEG_KEEP_RATIO})")


## DualTowerDataset 與資料增強

G9 的核心差異：title 與 content 分別 tokenize，各有獨立 token budget（64 + 256）。
增強邏輯（token dropout + content crop）與 G8 相同，確保只有架構不同。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MASK_TOKEN_ID = tokenizer.mask_token_id


def augment_content(content, crop_prob):
    """有 crop_prob 的機率從隨機位置截取 content，讓模型見到內文不同片段"""
    if not content or random.random() > crop_prob:
        return content
    words = content.split()
    if len(words) <= 1:
        return content
    start = random.randint(0, len(words) // 2)
    return " ".join(words[start:])


def apply_token_dropout(input_ids, attention_mask, mask_id, dropout_prob):
    """把非特殊 token 以 dropout_prob 的機率換成 <mask>，增強模型泛化"""
    special_ids = {
        tokenizer.cls_token_id, tokenizer.sep_token_id,
        tokenizer.pad_token_id, tokenizer.eos_token_id,
    }
    mask = (
        (torch.rand_like(input_ids.float()) < dropout_prob)
        & attention_mask.bool()
        & ~torch.isin(input_ids, torch.tensor(list(special_ids)))
    )
    input_ids[mask] = mask_id
    return input_ids


class DualTowerDataset(Dataset):
    """G9 雙塔 Dataset：title 與 content 分別 tokenize，各有獨立 token budget"""

    def __init__(self, df, tokenizer, augment=False):
        self.labels   = df["label"].tolist()
        self.titles   = df["title"].fillna("").tolist()
        self.contents = df["content"].fillna("").tolist()
        self.tokenizer = tokenizer
        self.augment   = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        title   = self.titles[idx]
        content = self.contents[idx]

        if self.augment:
            content = augment_content(content, CONTENT_CROP_PROB)

        enc_t = self.tokenizer(
            title,
            max_length=TITLE_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_c = self.tokenizer(
            content,
            max_length=CONTENT_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        t_ids, t_mask = enc_t["input_ids"].squeeze(0), enc_t["attention_mask"].squeeze(0)
        c_ids, c_mask = enc_c["input_ids"].squeeze(0), enc_c["attention_mask"].squeeze(0)

        if self.augment:
            t_ids = apply_token_dropout(t_ids.clone(), t_mask, MASK_TOKEN_ID, TOKEN_DROPOUT_PROB)
            c_ids = apply_token_dropout(c_ids.clone(), c_mask, MASK_TOKEN_ID, TOKEN_DROPOUT_PROB)

        return {
            "title_input_ids":        t_ids,
            "title_attention_mask":   t_mask,
            "content_input_ids":      c_ids,
            "content_attention_mask": c_mask,
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = DualTowerDataset(train_df, tokenizer, augment=True)
valid_dataset = DualTowerDataset(valid_df, tokenizer, augment=False)
test_dataset  = DualTowerDataset(test_df,  tokenizer, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=True)

print(f"train batches: {len(train_loader)} | valid batches: {len(valid_loader)} | test batches: {len(test_loader)}")

## 資料集基本測試

In [ ]:
sample_batch = next(iter(train_loader))
assert sample_batch["title_input_ids"].shape   == (BATCH_SIZE, TITLE_MAX_LEN),   f"title shape error"
assert sample_batch["title_attention_mask"].shape == (BATCH_SIZE, TITLE_MAX_LEN), f"title mask shape error"
assert sample_batch["content_input_ids"].shape == (BATCH_SIZE, CONTENT_MAX_LEN), f"content shape error"
assert sample_batch["content_attention_mask"].shape == (BATCH_SIZE, CONTENT_MAX_LEN), f"content mask shape error"
assert sample_batch["labels"].max().item() <= 1
assert sample_batch["labels"].min().item() >= 0
print("DualTowerDataset smoke test passed.")
print("title_input_ids shape:",   sample_batch["title_input_ids"].shape)
print("content_input_ids shape:", sample_batch["content_input_ids"].shape)

## DualTowerClassifier 模型定義（C1+C2+C3+C4）

**C1（常開）** 不對稱合併：`[h_t; h_c; h_t−h_c; h_c−h_t]` → 4×768 = 3072 維（與 v1 同維度，頭無需調整）。
方向性差異向量讓「標題有承諾、內文沒兌現」與其反向可區分。

**C2（常開）** title-aware attention pooling：以 title pooled 向量為 query、content token 為 key/value 加權求和，
凸顯「與標題承諾相關的內文 token」，避免 256 token mean-pool 稀釋廣告關鍵詞。

**C3（flag）** 一層 multi-head cross-attention：title token 為 Q、content token 為 K/V，
輸出「標題承諾在內文被回應的程度」向量，併入合併特徵。仍是雙塔（各自 encode），只在塔頂加對齊。

**C4（flag）** consistency loss：利用既有 423 組「同內文×不同標題→不同 label」對，
對合併表徵加 margin ranking loss 正則項（alpha=0.1），不需新資料。

In [ ]:
class TitleAwareAttentionPool(nn.Module):
    """C2：以 title pooled 向量為 query，對 content token 序列做 attention pooling。

    刻意用 title 而非純 learnable query：learnable query 只學 content salience，
    不一定與「標題承諾」相關；title query 凸顯「與標題承諾對應的內文 token」。
    """

    def __init__(self, hidden_size):
        super().__init__()
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.scale   = hidden_size ** -0.5

    def forward(self, h_title_pooled, content_hidden, content_mask):
        """
        h_title_pooled : (B, H)  — title mean-pool 向量
        content_hidden : (B, L, H)
        content_mask   : (B, L)  — 1=real token, 0=pad
        returns        : (B, H)  — title-attended content vector
        """
        q = self.q_proj(h_title_pooled).unsqueeze(1)   # (B, 1, H)
        k = self.k_proj(content_hidden)                  # (B, L, H)
        scores = (q * k).sum(-1) * self.scale            # (B, L)
        scores = scores.masked_fill(content_mask == 0, float("-inf"))
        attn   = scores.softmax(dim=-1)                  # (B, L)
        return (attn.unsqueeze(-1) * content_hidden).sum(1)  # (B, H)


class CrossAttentionAligner(nn.Module):
    """C3：一層 multi-head cross-attention（title token 為 Q, content token 為 K/V）。

    輸出：title 每個承諾點在內文被回應的程度，取 mean-pool 得到一個 H 維對齊向量。
    只做一層，不做深——已接近 cross-encoder 思路，深了主 test 更容易掉。
    關閉時（USE_CROSS_ATTENTION=False）forward 略過此層，合併特徵退回 C1+C2。
    """

    def __init__(self, hidden_size, num_heads=8, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(hidden_size, num_heads,
                                          dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, title_hidden, title_mask, content_hidden, content_mask):
        """
        title_hidden   : (B, Lt, H)
        title_mask     : (B, Lt)
        content_hidden : (B, Lc, H)
        content_mask   : (B, Lc)
        returns        : (B, H)  — mean-pooled cross-attended vector
        """
        # MultiheadAttention key_padding_mask: True 的位置被忽略
        content_key_mask = (content_mask == 0)   # (B, Lc), True = pad
        attn_out, _ = self.mha(
            query=title_hidden,
            key=content_hidden,
            value=content_hidden,
            key_padding_mask=content_key_mask,
        )
        attn_out = self.norm(attn_out + title_hidden)  # residual
        # mean-pool 非 pad 的 title token
        title_mask_f = title_mask.unsqueeze(-1).float()
        pooled = (attn_out * title_mask_f).sum(1) / title_mask_f.sum(1).clamp(min=1e-9)
        return pooled   # (B, H)


class DualTowerClassifier(nn.Module):
    """G9：C1+C2 常開，C3/C4 依 flag 啟用。

    合併維度：
      - C3 關：[h_t; h_c_attn; h_t−h_c_attn; h_c_attn−h_t] = 4×768 = 3072
      - C3 開：上述 4 項 + cross_align = 5×768 = 3840
    因此 C3 開關時分類頭 in_features 不同；flag 值在 __init__ 內決定，
    forward 關閉時略過 cross_align 相關計算，確保關閉與 C1+C2 baseline 完全等價。
    """

    def __init__(self, model_name, hidden_size=768, dropout=0.1):
        super().__init__()
        self.encoder      = AutoModel.from_pretrained(model_name)
        self.attn_pool    = TitleAwareAttentionPool(hidden_size)          # C2
        self.use_cross    = USE_CROSS_ATTENTION
        if self.use_cross:
            self.cross_align = CrossAttentionAligner(hidden_size, num_heads=8, dropout=dropout)  # C3
            merge_dim = hidden_size * 5
        else:
            merge_dim = hidden_size * 4
        self.classifier = nn.Sequential(
            nn.Linear(merge_dim, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def mean_pool(self, last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def encode_full(self, input_ids, attention_mask):
        """回傳 (pooled_vec, all_token_hidden)"""
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        pooled = self.mean_pool(hidden, attention_mask)
        return pooled, hidden

    def _merge(self, h_t, h_t_hidden, t_mask, h_c_hidden, c_mask):
        """組合合併向量（C1+C2+C3，依 flag）。"""
        # C2：title-aware attention pool of content
        h_c_attn = self.attn_pool(h_t, h_c_hidden, c_mask)   # (B, H)
        # C1：不對稱合併（有號方向）
        parts = [h_t, h_c_attn, h_t - h_c_attn, h_c_attn - h_t]
        if self.use_cross:
            # C3：一層 cross-attention 對齊
            align = self.cross_align(h_t_hidden, t_mask, h_c_hidden, c_mask)  # (B, H)
            parts.append(align)
        return torch.cat(parts, dim=-1)

    def forward(self, title_input_ids, title_attention_mask,
                content_input_ids, content_attention_mask, labels=None):
        h_t, h_t_hidden = self.encode_full(title_input_ids,   title_attention_mask)
        _,   h_c_hidden = self.encode_full(content_input_ids, content_attention_mask)
        v      = self._merge(h_t, h_t_hidden, title_attention_mask, h_c_hidden, content_attention_mask)
        logits = self.classifier(v)
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return logits, loss

    def forward_for_consistency(self, title_input_ids, title_attention_mask,
                                content_input_ids, content_attention_mask):
        """C4：回傳合併表徵（不過分類頭），供 consistency loss 計算。"""
        h_t, h_t_hidden = self.encode_full(title_input_ids,   title_attention_mask)
        _,   h_c_hidden = self.encode_full(content_input_ids, content_attention_mask)
        return self._merge(h_t, h_t_hidden, title_attention_mask, h_c_hidden, content_attention_mask)


model = DualTowerClassifier(MODEL_NAME)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
merge_dim = 768 * 5 if USE_CROSS_ATTENTION else 768 * 4
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")
print(f"USE_CROSS_ATTENTION={USE_CROSS_ATTENTION} | merge_dim={merge_dim}")
print(f"USE_CONSISTENCY_LOSS={USE_CONSISTENCY_LOSS}")

## 訓練迴圈

In [ ]:
# ── C4：consistency loss 用的對比對（同內文×不同標題→不同 label）────────────
# 利用 adversarial_g8.csv 中 113 組 tone 對：同一篇內文，orig 標題(label=1, clickbait)
# vs plain 標題(label=0, 直白)。類型編碼在 id 後綴（_orig / _plain），非獨立欄位。
# 已查證：113 組、orig 全 label=1、plain 全 label=0、content 100% 相同。
# 只在 USE_CONSISTENCY_LOSS=True 時建 loader；關閉時不影響主訓練流程。

if USE_CONSISTENCY_LOSS:
    adv_raw_path = PROCESSED_DIR / "adversarial_g8.csv"
    adv_raw = pd.read_csv(adv_raw_path)

    # 類型在 id 後綴：_orig（clickbait 標題, label=1）、_plain（直白標題, label=0）
    orig_df  = adv_raw[adv_raw["id"].str.endswith("_orig")].copy()
    plain_df = adv_raw[adv_raw["id"].str.endswith("_plain")].copy()

    # 配對 key：去掉 _orig / _plain 後綴（同一篇文章的 orig 與 plain 共享 key）
    orig_df["pair_key"]  = orig_df["id"].str.replace(r"_orig$",  "", regex=True)
    plain_df["pair_key"] = plain_df["id"].str.replace(r"_plain$", "", regex=True)

    pairs = orig_df.merge(plain_df, on="pair_key", suffixes=("_pos", "_neg"))
    # 確保 label 方向正確：pos=1（clickbait 標題），neg=0（直白標題），共享 content
    pairs = pairs[(pairs["label_pos"] == 1) & (pairs["label_neg"] == 0)].reset_index(drop=True)
    # 健全性檢查：pos/neg 的 content 應完全相同（C4 的「同內文」前提）
    same_content = (pairs["content_pos"] == pairs["content_neg"]).mean()
    assert same_content == 1.0, f"tone 對 content 不一致（{same_content:.1%}），C4 前提不成立"
    print(f"C4 consistency pairs: {len(pairs)} 組（content 同內文比例 {same_content:.0%}）")

    class ConsistencyPairDataset(Dataset):
        """同內文×不同標題對，各自 tokenize，批次內 (pos, neg) 成對。"""
        def __init__(self, pairs_df, tok):
            self.titles_pos  = pairs_df["title_pos"].fillna("").tolist()
            self.titles_neg  = pairs_df["title_neg"].fillna("").tolist()
            self.contents    = pairs_df["content_pos"].fillna("").tolist()
            self.tok = tok

        def __len__(self):
            return len(self.titles_pos)

        def _enc(self, text, max_len):
            return self.tok(text, max_length=max_len, padding="max_length",
                            truncation=True, return_tensors="pt")

        def __getitem__(self, idx):
            enc_tp = self._enc(self.titles_pos[idx], TITLE_MAX_LEN)
            enc_tn = self._enc(self.titles_neg[idx], TITLE_MAX_LEN)
            enc_c  = self._enc(self.contents[idx], CONTENT_MAX_LEN)
            return {
                "pos_title_input_ids":        enc_tp["input_ids"].squeeze(0),
                "pos_title_attention_mask":   enc_tp["attention_mask"].squeeze(0),
                "neg_title_input_ids":        enc_tn["input_ids"].squeeze(0),
                "neg_title_attention_mask":   enc_tn["attention_mask"].squeeze(0),
                "content_input_ids":          enc_c["input_ids"].squeeze(0),
                "content_attention_mask":     enc_c["attention_mask"].squeeze(0),
            }

    consist_dataset = ConsistencyPairDataset(pairs, tokenizer)
    consist_loader  = DataLoader(consist_dataset, batch_size=BATCH_SIZE,
                                 shuffle=True, num_workers=2, pin_memory=True)
    consist_iter = iter(consist_loader)
    print(f"ConsistencyPairDataset: {len(consist_dataset)} pairs | {len(consist_loader)} batches")
else:
    consist_loader = None
    consist_iter   = None
    print("USE_CONSISTENCY_LOSS=False — C4 loader skipped.")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for batch in loader:
            title_input_ids        = batch["title_input_ids"].to(device)
            title_attention_mask   = batch["title_attention_mask"].to(device)
            content_input_ids      = batch["content_input_ids"].to(device)
            content_attention_mask = batch["content_attention_mask"].to(device)
            labels                 = batch["labels"].to(device)

            logits, loss = model(
                title_input_ids=title_input_ids,
                title_attention_mask=title_attention_mask,
                content_input_ids=content_input_ids,
                content_attention_mask=content_attention_mask,
                labels=labels,
            )
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=-1)[:, 1]
            preds = (probs >= THRESHOLD).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1


def consistency_loss_step(model, batch, device):
    """C4：margin ranking loss。content encode 一次、pos/neg title 各一次（3 次塔 forward）。

    梯度設計：content 塔保留梯度，使 C4 也能訓練 content encoder，
    與 attn_pool/cross_align 的 content 側——這正是 C4 想強化「模型看內文」的目的

    注意：content hidden 共用於 pos/neg 兩條 _merge，兩者的 loss 合併後一次 backward，
    屬同一張計算圖，不會觸發 backward-twice 錯誤。
    """
    pos_t_ids  = batch["pos_title_input_ids"].to(device)
    pos_t_mask = batch["pos_title_attention_mask"].to(device)
    neg_t_ids  = batch["neg_title_input_ids"].to(device)
    neg_t_mask = batch["neg_title_attention_mask"].to(device)
    c_ids  = batch["content_input_ids"].to(device)
    c_mask = batch["content_attention_mask"].to(device)

    # content 塔跑一次（pos/neg 共享相同內文），保留梯度
    _, h_c_hid = model.encode_full(c_ids, c_mask)

    # pos title 塔
    h_tp, h_tp_hid = model.encode_full(pos_t_ids, pos_t_mask)
    v_pos  = model._merge(h_tp, h_tp_hid, pos_t_mask, h_c_hid, c_mask)
    score_pos = model.classifier(v_pos)[:, 1]

    # neg title 塔
    h_tn, h_tn_hid = model.encode_full(neg_t_ids, neg_t_mask)
    v_neg  = model._merge(h_tn, h_tn_hid, neg_t_mask, h_c_hid, c_mask)
    score_neg = model.classifier(v_neg)[:, 1]

    # hinge：max(0, margin - (score_pos - score_neg))
    return torch.relu(0.5 - (score_pos - score_neg)).mean()


MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_valid_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for step, batch in enumerate(train_loader, 1):
        title_input_ids        = batch["title_input_ids"].to(DEVICE)
        title_attention_mask   = batch["title_attention_mask"].to(DEVICE)
        content_input_ids      = batch["content_input_ids"].to(DEVICE)
        content_attention_mask = batch["content_attention_mask"].to(DEVICE)
        labels                 = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits, ce_loss = model(
            title_input_ids=title_input_ids,
            title_attention_mask=title_attention_mask,
            content_input_ids=content_input_ids,
            content_attention_mask=content_attention_mask,
            labels=labels,
        )

        # C4：consistency loss（依 flag）
        if USE_CONSISTENCY_LOSS and consist_iter is not None:
            try:
                cb = next(consist_iter)
            except StopIteration:
                consist_iter = iter(consist_loader)
                cb = next(consist_iter)
            c_loss = consistency_loss_step(model, cb, DEVICE)
            loss = ce_loss + CONSISTENCY_LOSS_ALPHA * c_loss
        else:
            loss = ce_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss += ce_loss.item()
        if step % 500 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch} step {step}/{len(train_loader)} | ce_loss={train_loss/step:.4f} | lr={current_lr:.2e}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = evaluate(model, valid_loader, DEVICE)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch} | train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | lr={current_lr:.2e}")

    if val_f1 > best_valid_f1:
        best_valid_f1 = val_f1
        model.encoder.save_pretrained(MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
        torch.save(model.classifier.state_dict(),
                   MODEL_OUTPUT_DIR / "classifier_head.pt")
        torch.save(model.attn_pool.state_dict(),
                   MODEL_OUTPUT_DIR / "attn_pool.pt")
        if USE_CROSS_ATTENTION:
            torch.save(model.cross_align.state_dict(),
                       MODEL_OUTPUT_DIR / "cross_align.pt")
        flag_cfg = {
            "USE_CROSS_ATTENTION": USE_CROSS_ATTENTION,
            "USE_CONSISTENCY_LOSS": USE_CONSISTENCY_LOSS,
            "run_tag": RUN_TAG,
        }
        import json as _json
        (MODEL_OUTPUT_DIR / "g9_config.json").write_text(
            _json.dumps(flag_cfg, indent=2), encoding="utf-8")
        print(f"  -> Best model saved (val_f1={val_f1:.4f})")

print(f"\nTraining done. Best val F1: {best_valid_f1:.4f}")

## 訓練曲線

In [ ]:
RESULTS_DIR = DRIVE_PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

epochs_range = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
ax1.plot(epochs_range, history["val_loss"],   marker="o", label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Train / Val Loss (G9)"); ax1.legend(); ax1.grid(True)

ax2.plot(epochs_range, history["val_f1"],  marker="o", label="Val Macro F1", color="green")
ax2.plot(epochs_range, history["val_acc"], marker="o", label="Val Accuracy",  color="orange")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title("Val F1 & Accuracy (G9)"); ax2.legend(); ax2.grid(True)

fig.tight_layout()
curve_path = RESULTS_DIR / f"{RUN_TAG}_transformer_training_curve.png"
fig.savefig(curve_path, dpi=150)
plt.show()
print("Training curve saved to:", curve_path)

## 測試集評估

In [ ]:
# 重新載入最佳 checkpoint（從 g9_config.json 讀 flag，不依賴全域 USE_CROSS_ATTENTION）
import json as _json
_cfg = _json.loads((MODEL_OUTPUT_DIR / "g9_config.json").read_text(encoding="utf-8"))
_use_cross = _cfg.get("USE_CROSS_ATTENTION", False)

best_model = DualTowerClassifier(MODEL_NAME)
# 若 _use_cross 與全域 flag 不符，修正模型結構
if _use_cross != best_model.use_cross:
    if _use_cross:
        best_model.cross_align = CrossAttentionAligner(768, num_heads=8, dropout=0.1)
    elif hasattr(best_model, "cross_align"):
        del best_model.cross_align
    best_model.use_cross = _use_cross
    # 重建分類頭（維度隨 use_cross 而定，須與 checkpoint 對應）
    merge_dim = 768 * 5 if _use_cross else 768 * 4
    best_model.classifier = nn.Sequential(
        nn.Linear(merge_dim, 768), nn.Tanh(), nn.Dropout(0.1), nn.Linear(768, 2)
    )

best_model.encoder = AutoModel.from_pretrained(MODEL_OUTPUT_DIR)
best_model.classifier.load_state_dict(
    torch.load(MODEL_OUTPUT_DIR / "classifier_head.pt", map_location=DEVICE)
)
best_model.attn_pool.load_state_dict(
    torch.load(MODEL_OUTPUT_DIR / "attn_pool.pt", map_location=DEVICE)
)
if _use_cross:
    best_model.cross_align.load_state_dict(
        torch.load(MODEL_OUTPUT_DIR / "cross_align.pt", map_location=DEVICE)
    )
best_model = best_model.to(DEVICE)
best_model.eval()
print(f"Loaded checkpoint: USE_CROSS_ATTENTION={_use_cross}")

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        logits, _ = best_model(
            title_input_ids=batch["title_input_ids"].to(DEVICE),
            title_attention_mask=batch["title_attention_mask"].to(DEVICE),
            content_input_ids=batch["content_input_ids"].to(DEVICE),
            content_attention_mask=batch["content_attention_mask"].to(DEVICE),
        )
        probs = torch.softmax(logits, dim=-1)[:, 1]
        preds = (probs >= THRESHOLD).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("=== Test Set Results (G9) ===")
print(classification_report(all_labels, all_preds, target_names=["non-clickbait", "clickbait"]))
print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds))

## 儲存評估結果

In [ ]:
RESULTS_DIR = DRIVE_PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Classification report → CSV ---
report_dict = classification_report(
    all_labels, all_preds,
    target_names=["non-clickbait", "clickbait"],
    output_dict=True,
)
report_df = pd.DataFrame(report_dict).transpose()
report_path = RESULTS_DIR / f"{RUN_TAG}_transformer_metrics.csv"
report_df.to_csv(report_path, encoding="utf-8-sig")
print("Metrics saved to:", report_path)
print(report_df.round(4))

# --- Metrics JSON（accuracy + macro_f1 + flag config 可追溯）---
metrics_json = {
    "run_tag": RUN_TAG,
    "accuracy": report_dict["accuracy"],
    "macro_f1": report_dict["macro avg"]["f1-score"],
    "USE_CROSS_ATTENTION": USE_CROSS_ATTENTION,
    "USE_CONSISTENCY_LOSS": USE_CONSISTENCY_LOSS,
}
with open(METRICS_OUT, "w") as f:
    json.dump(metrics_json, f, indent=2)
print("JSON metrics saved to:", METRICS_OUT)

# --- Confusion matrix → PNG ---
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(["non-clickbait", "clickbait"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["non-clickbait", "clickbait"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"XLM-RoBERTa Confusion Matrix ({RUN_TAG.upper()} DualTower)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
cm_path = RESULTS_DIR / f"{RUN_TAG}_transformer_confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.show()
print("Confusion matrix saved to:", cm_path)

## Task：驗證 B 複刻（內文利用率測試）

固定標題、換不同內文，觀察 G8（單塔）vs G9（雙塔）的預測機率是否隨內文改變——
G9 的機率分散度若大於 G8，本身就是 content utilization 改善的直接證據。

**兩層分析**：

1. **Within-title group**（主要指標）
   固定同一個誘導標題（`TITLE_BC`），換 3 種內文：兌現 / 廣告 / 空。
   在此組內算 std，排除 title sensitivity 干擾。比較 G8 std vs G9 std。

2. **Cross-title cases**（原始 4 case，保留作對照）
   Case A 是 title 消融對比（不同標題），不是 within-title 測試；
   Case D 是 sanity check。
   此組 std 含 title sensitivity，不單獨作為 content utilization 的證據。

**方向判斷**：由 ground-truth label 決定，不套用「廣告/空內文 → p 下降」的錯誤假設。
誘導標題 + 廣告/空內文 = label 1（典型 clickbait），G9 若學對應維持或增強 p(clickbait)。

此 cell **不依賴訓練 loop**，可單獨執行：
- 需先執行測試集評估 cell 取得 `best_model`（G9）
- helpers cell 會自動載入 G8 checkpoint（`models/xlm-roberta-clickbait-g8`）以產生對照
- 本 cell 已自包含 G8 vs G9 std 比較，不再依賴外部 `explain_tokens.py`

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification


def predict_g9(title, content, model, tokenizer, device):
    """回傳 G9 雙塔對 (title, content) 預測為 clickbait 的機率"""
    model.eval()
    with torch.no_grad():
        enc_t = tokenizer(
            title,
            max_length=TITLE_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_c = tokenizer(
            content,
            max_length=CONTENT_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        logits, _ = model(
            title_input_ids=enc_t["input_ids"].to(device),
            title_attention_mask=enc_t["attention_mask"].to(device),
            content_input_ids=enc_c["input_ids"].to(device),
            content_attention_mask=enc_c["attention_mask"].to(device),
        )
        return F.softmax(logits, dim=-1)[0, 1].item()


# ── G8 單塔對照 ──────────────────────────────────────────────────────
G8_MODEL_DIR = DRIVE_PROJECT / "models" / "xlm-roberta-clickbait-g8"
G8_MAX_LENGTH = 256

g8_model = AutoModelForSequenceClassification.from_pretrained(G8_MODEL_DIR).to(DEVICE)
g8_model.eval()
g8_tokenizer = AutoTokenizer.from_pretrained(G8_MODEL_DIR)


def predict_g8(title, content, model=g8_model, tokenizer=g8_tokenizer, device=DEVICE):
    model.eval()
    with torch.no_grad():
        enc = tokenizer(
            title, content,
            max_length=G8_MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
        return F.softmax(logits, dim=-1)[0, 1].item()


print("predict_g9 and predict_g8 defined.")

In [ ]:
TITLE_BC = "探討優質睡眠與深層放鬆的生理機制：特定種類的植物精油成分有助於副交感神經的穩定"
CONTENT_FULFILLED = (
    "研究顯示，薰衣草中的沉香醇（linalool）可透過抑制興奮性神經傳遞物質、"
    "促進 GABA 受體活性，進而降低心跳速率並穩定副交感神經系統，改善入睡效率。"
    "實驗以雙盲設計追蹤120名受試者四週，薰衣草組平均入睡時間縮短14分鐘。"
)
CONTENT_AD = (
    "本產品採用天然植物精華配方，經過嚴格品質把關，讓您每天都能感受到煥然一新的活力。"
    "立即點擊下方連結，享受限時85折優惠，加入LINE好友即可獲得精美試用包。"
    "全館滿999免運，現在下單再享買二送一優惠，數量有限，售完為止！"
)
CONTENT_EMPTY = ""

# ── Part 1：Within-title group（Bug4-A/B/C 指標，固定標題換內文）──────
within_title_cases = [
    (CONTENT_FULFILLED, 0, "Bug4-C（誘導標題 + 兌現內文）→ gt=0，p 應低"),
    (CONTENT_AD,        1, "Bug4-A（誘導標題 + 廣告軟文）→ gt=1，p 應高（主要指標）"),
    (CONTENT_EMPTY,     1, "Bug4-B（誘導標題 + 空內文）  → gt=1，stress test"),
]

print("=" * 70)
print("【Part 1】Within-title group（固定標題，換內文）— Bug 4 探針")
print(f"  Title: {TITLE_BC}")
print("=" * 70)

g8_wt, g9_wt = [], []
for content, gt, desc in within_title_cases:
    p8 = predict_g8(TITLE_BC, content)
    p9 = predict_g9(TITLE_BC, content, best_model, tokenizer, DEVICE)
    g8_wt.append(p8)
    g9_wt.append(p9)
    ok8 = (p8 > THRESHOLD) == bool(gt)
    ok9 = (p9 > THRESHOLD) == bool(gt)
    flip = "翻轉✓" if ok9 and not ok8 else ""
    print(f"  [{desc}]")
    print(f"    G8 p={p8:.4f} {'✓' if ok8 else '✗'}   G9 p={p9:.4f} {'✓' if ok9 else '✗'}  {flip}")

g8_wt_std = float(np.std(g8_wt))
g9_wt_std = float(np.std(g9_wt))
print(f"\n  G8 within-title std:   {g8_wt_std:.4f}")
print(f"  G9 within-title std: {g9_wt_std:.4f}")
print(f"  {'✓ G9 std > G8（content utilization 改善）' if g9_wt_std > g8_wt_std else '✗ G9 未超過 G8'}")
print(f"\n  Bug4-A p (廣告軟文): G8={g8_wt[1]:.4f}  G9={g9_wt[1]:.4f}")
print(f"  Bug4-C p (兌現內文): G8={g8_wt[0]:.4f}  G9={g9_wt[0]:.4f}")

# ── Part 2：Cross-title cases ─────────────────────────────────────────
TITLE_PLAIN_ZH = "薰衣草精油可改善睡眠品質，研究：與副交感神經活化有關"
TITLE_INDUCED_EN = (
    "Examining the Neurological Basis of Deep Rest: "
    "A Specific Category of Plant-derived Compounds May Enhance Parasympathetic Activity"
)
CONTENT_AD_EN = (
    "Try our premium sleep supplement today! Clinically inspired formula. "
    "Click the link below for a limited-time 20% discount. "
    "Subscribe now and get a free trial pack with your first order."
)

cross_title_cases = [
    ("探討優質睡眠的生理機制：薰衣草精油成分有助於副交感神經穩定",
     CONTENT_FULFILLED, 0, "Case A：title 消融（直白標題 + 兌現內文）→ gt=0"),
    (TITLE_BC, CONTENT_AD,    1, "Case B：誘導標題 + 廣告軟文 → gt=1（Bug4-A）"),
    (TITLE_BC, CONTENT_EMPTY, 1, "Case C：誘導標題 + 空內文 → gt=1（Bug4-B）"),
    (TITLE_PLAIN_ZH,
     "研究顯示，薰衣草中的沉香醇可透過抑制興奮性神經傳遞物質來穩定副交感神經。",
     0, "Case D：sanity check（平實標題 + 兌現內文）→ gt=0"),
    (TITLE_INDUCED_EN, CONTENT_AD_EN, 1,
     "Case E：英文誘導標題 + 英文廣告內文 → gt=1（Bug4-A 英文版）"),
]

print("\n" + "=" * 70)
print("【Part 2】Cross-title cases（std 含 title sensitivity，參考用）")
print("=" * 70)

g8_ct, g9_ct = [], []
for title, content, gt, desc in cross_title_cases:
    p8 = predict_g8(title, content)
    p9 = predict_g9(title, content, best_model, tokenizer, DEVICE)
    g8_ct.append(p8)
    g9_ct.append(p9)
    ok8 = (p8 > 0.5) == bool(gt)
    ok9 = (p9 > 0.5) == bool(gt)
    print(f"  [{desc}]")
    print(f"    G8 p={p8:.4f} {'✓' if ok8 else '✗'}   G9 p={p9:.4f} {'✓' if ok9 else '✗'}")

g8_ct_std = float(np.std(g8_ct))
g9_ct_std = float(np.std(g9_ct))

print("\n" + "=" * 70)
print(f"[Summary] G8 vs G9 — config: USE_CROSS_ATTENTION={USE_CROSS_ATTENTION}")
print(f"  Within-title std（排除 title sensitivity）:")
print(f"    G8={g8_wt_std:.4f}  G9={g9_wt_std:.4f}  "
      f"{'✓ G9 > G8' if g9_wt_std > g8_wt_std else '✗ 未超過 G8'}")
print(f"  Cross-title std（參考用）:")
print(f"    G8={g8_ct_std:.4f}  G9={g9_ct_std:.4f}")
print("=" * 70)